# Manufacturing Quality Analytics
### Investigating Drivers of Defect Rate in Production Data

**Objective:** Identify which operational factors most strongly predict product defects, and provide a data-backed recommendation for where quality-improvement effort should be focused.

**Dataset:** 3,240 production records, 17 operational variables (production volume, maintenance hours, supplier quality, etc.), sourced from a synthetic manufacturing dataset.


## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
from sqlalchemy import create_engine
import urllib
import os


PASSWORD = os.environ.get("AZURE_SQL_PASSWORD")

odbc_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=mfgabtesting.database.windows.net;"
    "DATABASE=free-sql-db-2847881;"
    "UID=mfgadmin;"
    f"PWD={PASSWORD};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;"
)
params = urllib.parse.quote_plus(odbc_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

df = pd.read_sql("SELECT * FROM manufacturing_defects", engine)
df.shape

## 2. Initial Exploration

Before analyzing anything, get a basic understanding of the data: what columns exist, their types, whether there's missing data, and the general shape of each variable.

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Check for missing values and duplicate rows
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

**Observation:** No missing values, no duplicates — the dataset is clean at the structural level. 3,240 rows, 17 columns, target variable is `DefectStatus` (binary: 1 = high defect, 0 = low defect).

Checking the target variable's balance:

In [ ]:
df['DefectStatus'].value_counts(normalize=True)

**Observation:** 84% of records are flagged as high-defect. This is unusually high for a real production line, which is a sign this dataset may be synthetic rather than a real operational log — worth keeping in mind when interpreting results (see Data Quality Notes at the end).

## 3. Data Quality Audit — Which Variables Actually Matter?

With 17 columns, not all of them are likely to be meaningfully related to defects. Rather than analyzing every column with equal effort, start by screening for the variables that actually show a relationship with the target — this focuses the rest of the analysis on what matters.

In [ ]:
correlations = df.corr(numeric_only=True)['DefectStatus'].sort_values(key=abs, ascending=False)
correlations

**Observation:** Most variables show near-zero correlation with `DefectStatus` — consistent with a synthetic dataset where many columns are independently generated "noise" fields rather than causally connected to the outcome.

Four variables stand out:
- **MaintenanceHours** (0.297) — the strongest signal
- **DefectRate** (0.246) — likely the continuous metric `DefectStatus` was derived/binned from; excluded as a predictor to avoid data leakage
- **QualityScore** (-0.199) — negative correlation (higher quality score → fewer defects, as expected), but somewhat circular as a "finding"
- **ProductionVolume** (0.129) — weaker on its own, but worth checking for interaction effects later

**Decision:** Focus the investigation on `MaintenanceHours` as the primary driver, since it's the strongest, most operationally actionable signal — and it's not just restating the target in different words the way QualityScore or DefectRate would.

## 4. Deep Dive: Maintenance Hours

### 4.1 Understanding the Distribution

Before splitting the data into groups, look at the shape of `MaintenanceHours` itself — this informs how to split it fairly.

In [ ]:
mean_val = df['MaintenanceHours'].mean()
median_val = df['MaintenanceHours'].median()
print(f"Mean: {mean_val:.2f}, Median: {median_val:.2f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(df['MaintenanceHours'], vert=True, patch_artist=True,
           boxprops=dict(facecolor='#8b7fd6'))
ax.axhline(mean_val, color='red', linestyle='--', label=f'Mean = {mean_val:.2f}')
ax.axhline(median_val, color='blue', linestyle='-', label=f'Median = {median_val:.2f}')
ax.set_ylabel('Maintenance Hours')
ax.set_title('Distribution of MaintenanceHours')
ax.legend()
plt.show()

**Observation:** The distribution is fairly symmetric with no significant outliers (bounded 0-24 range). Mean and median are close (11.48 vs 12.00).

**Decision:** Split at the **median**, not the mean. Median guarantees a balanced ~50/50 group split by definition, which gives more reliable statistical power for the test that follows — this is the more defensible default even when, as here, mean and median happen to be close.

### 4.2 Splitting Into Comparison Groups

In [ ]:
median_maint = df['MaintenanceHours'].median()

low_maint = df[df['MaintenanceHours'] <= median_maint]
high_maint = df[df['MaintenanceHours'] > median_maint]

rate_low = low_maint['DefectStatus'].mean()
rate_high = high_maint['DefectStatus'].mean()

print(f"Low maintenance (<= {median_maint}h): n={len(low_maint)}, defect rate={rate_low:.1%}")
print(f"High maintenance (> {median_maint}h): n={len(high_maint)}, defect rate={rate_high:.1%}")
print(f"Difference: {(rate_high-rate_low)*100:.1f} percentage points")

**Observation:** A 22-point gap in defect rate between the two groups. Before concluding this is meaningful, it needs to be tested — a gap this size could theoretically arise from random chance in how the data happened to split.

### 4.3 Statistical Significance — Two-Proportion Z-Test

`DefectStatus` is a binary outcome, and the comparison is between **proportions** (% defective in each group) — this is exactly the scenario a two-proportion z-test is designed for. (A chi-square test would give an equivalent result for this 2×2 comparison; the z-test additionally provides a confidence interval on the size of the difference, which is more useful for a business recommendation.)

In [ ]:
count = np.array([high_maint['DefectStatus'].sum(), low_maint['DefectStatus'].sum()])
nobs = np.array([len(high_maint), len(low_maint)])

zstat, pval = proportions_ztest(count, nobs)

se = np.sqrt(rate_low*(1-rate_low)/len(low_maint) + rate_high*(1-rate_high)/len(high_maint))
diff = rate_high - rate_low
ci_low, ci_high = diff - 1.96*se, diff + 1.96*se

print(f"Z-statistic: {zstat:.2f}")
print(f"P-value: {pval:.6f}")
print(f"95% CI for difference: ({ci_low*100:.1f}, {ci_high*100:.1f}) percentage points")

**Finding:** The 22-point gap is statistically significant (p < 0.001) — this is not random noise. With 95% confidence, the true difference in defect rate between low and high maintenance groups falls between 19.8 and 24.4 percentage points.

## 5. Is the Relationship Linear? Granular Binning

Knowing the effect is real, the next natural question: **does defect rate rise steadily with maintenance hours, or is there a specific point where risk jumps?** This matters for the recommendation — a threshold implies a specific intervention point, while a linear relationship would imply "always reduce maintenance hours," which may not be the right message.

In [ ]:
bins = list(range(0, 25, 2))
df['maint_bin'] = pd.cut(df['MaintenanceHours'], bins=bins, include_lowest=True)
bin_summary = df.groupby('maint_bin', observed=True)['DefectStatus'].agg(['mean','count'])
bin_summary['mean'] = (bin_summary['mean']*100).round(1)
bin_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(bin_summary.index.astype(str), bin_summary['mean'], color='#8b7fd6')
ax.set_xlabel('Maintenance Hours Bin')
ax.set_ylabel('Defect Rate (%)')
ax.set_title('Defect Rate by Maintenance Hours — Threshold Effect')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Finding:** The relationship is **not linear** — it's a threshold effect. Defect rate holds steady around 63-75% up to ~10 maintenance hours, then jumps sharply to 94-97% from 12 hours onward.

**Interpretation:** This suggests equipment crossing the ~10-12 hour maintenance threshold is likely already degraded — maintenance hours may be a *symptom* of underlying equipment condition rather than a *cause* of defects.

**Robustness check:** re-running this at 1-hour bin resolution (instead of 2-hour) produces the same threshold location (~10-11 hours), confirming this pattern isn't an artifact of the bin width chosen.

## 6. Does Production Volume Change the Picture?

`ProductionVolume` and `SupplierQuality` also showed some correlation with defects, though weaker individually. Rather than treating them as two separate, independent factors, check whether they **interact** — does supplier quality's effect on defects depend on how much volume is being produced?

In [ ]:
def volume_bin(v):
    if v < 300: return 'Low (<300)'
    elif v < 600: return 'Medium (300-600)'
    elif v < 900: return 'High (600-900)'
    else: return 'Very High (900+)'

def quality_bin(q):
    if q < 85: return 'Low (80-85)'
    elif q < 90: return 'Medium (85-90)'
    elif q < 95: return 'High (90-95)'
    else: return 'Very High (95-100)'

df['vol_bin'] = df['ProductionVolume'].apply(volume_bin)
df['qual_bin'] = df['SupplierQuality'].apply(quality_bin)

matrix = df.pivot_table(values='DefectStatus', index='vol_bin', columns='qual_bin', aggfunc='mean')
(matrix * 100).round(1)

**Finding:** Supplier quality's protective effect is only visible at Low/Medium production volume (defect rate varies 77-87% across quality tiers). At **Very High** production volume, defect rate stays elevated (96-98%) **regardless of supplier quality** — volume appears to overwhelm supplier quality's influence at scale.

## 7. Conclusions & Recommendation

**Key Findings:**
1. Maintenance hours is the strongest single predictor of defect rate in this dataset, but the relationship is a **threshold effect**, not linear — risk jumps sharply once equipment crosses ~10-12 maintenance hours (a weekly figure).
2. This threshold effect is statistically significant (two-proportion z-test, p < 0.001, 95% CI 19.8-24.4pp) and robust to bin-width choice.
3. Production volume interacts with supplier quality — at very high volume, defect rate stays elevated regardless of supplier quality, suggesting volume-driven strain may outweigh input quality at scale.

**Recommendation:** Prioritize investigating equipment crossing the 10-12 hour weekly maintenance threshold, particularly during high-volume production runs — this is the highest-risk combination observed. Maintenance hours should not be treated as a simple linear lever to reduce; the underlying equipment condition driving high maintenance needs is the more likely root cause.

**Data Quality Notes:**
- This is a synthetic dataset (Kaggle). An internal consistency check found most variables show near-zero correlation with each other, consistent with independently-generated fields rather than a fully realistic causal simulation.
- Documented units are inconsistent across columns (e.g. ProductionVolume is "per day," MaintenanceHours is "per week," SafetyIncidents is "per month") — this rules out treating the data as a coherent time series. All analysis here is cross-sectional (comparing groups within the dataset), not time-series based.
- The overall 84% defect rate is unusually high for a real production line and should not be interpreted as representative of real-world defect rates — the focus of this analysis is on *relative* risk factors, not absolute rates.
